# DEER Spectroscopy Simulation

<a href="https://colab.research.google.com/github/elkins/diff-epr/blob/main/examples/interactive_tutorials/deer_spectroscopy_tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Double Electron-Electron Resonance (DEER) is a pulsed EPR technique used to measure distances between unpaired electrons (typically spin labels attached to a protein) in the 1.5 to 8 nm range.

The dipolar coupling frequency $\nu_{dd}$ depends on the distance $r$ between the spins:
$$ \nu_{dd} = \frac{52040}{r^3} \text{ MHz} $$

The DEER time-domain signal $V(t)$ exhibits oscillations corresponding to this frequency, damped by a background decay $k_{bg}$ due to intermolecular interactions:
$$ V(t) = \left[1 - \lambda (1 - \cos(2\pi \nu_{dd} t))\right] \exp(-k_{bg} t) $$
where $\lambda$ is the modulation depth.

This tutorial demonstrates how to simulate DEER signals using the `diff_epr` physics module.

In [ ]:
import sys

if "google.colab" in sys.modules:
    !pip install -q diff-epr matplotlib biotite
else:
    sys.path.append("../../")

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

## 1. Distance Distribution $P(r)$

Spin labels attached to proteins are highly flexible. Instead of a single discrete distance, we measure a distribution of distances $P(r)$. We'll create a synthetic bimodal Gaussian distance distribution.

In [ ]:
from diff_epr.kernels import deer_trace

# Generate a synthetic bimodal distance distribution
r = jnp.linspace(15.0, 60.0, 500)  # Distances in Angstroms

# Mode 1: centered at 30A
p1 = jnp.exp(-0.5 * ((r - 30.0) / 3.0) ** 2)
# Mode 2: centered at 45A
p2 = 0.6 * jnp.exp(-0.5 * ((r - 45.0) / 4.0) ** 2)

P_r = p1 + p2
P_r = P_r / jnp.sum(P_r)  # Normalize

plt.figure(figsize=(8, 4))
plt.plot(np.array(r), np.array(P_r), linewidth=2, color="darkgreen")
plt.xlabel("Distance $r$ (Å)", fontsize=12)
plt.ylabel("Probability $P(r)$", fontsize=12)
plt.title("Spin-Label Distance Distribution", fontsize=14)
plt.tight_layout()
plt.show()

## 2. DEER Time Traces

Let's simulate the DEER traces $V(t)$ for single discrete distances vs. the full distribution.

In [ ]:
# Time axis (microseconds)
t = jnp.linspace(0.0, 5.0, 500)

# Single distance trace for r = 30 Angstroms
v_single_30 = deer_trace(jnp.array([30.0]), t, modulation_depth=0.3, background_decay=0.05)

# Single distance trace for r = 45 Angstroms
v_single_45 = deer_trace(jnp.array([45.0]), t, modulation_depth=0.3, background_decay=0.05)

# Full distribution trace (weighted average of all distances in P(r))
# To do this correctly, we simulate traces for all distances, then take a weighted sum
v_dist_all = deer_trace(
    r, t, modulation_depth=0.3, background_decay=0.0
)  # Background applied later
v_dist = jnp.sum(v_dist_all * P_r[:, None], axis=0) * jnp.exp(-0.05 * t)

plt.figure(figsize=(10, 6))
plt.plot(np.array(t), np.array(v_single_30), label="Single Distance (r=30Å)", alpha=0.6)
plt.plot(np.array(t), np.array(v_single_45), label="Single Distance (r=45Å)", alpha=0.6)
plt.plot(np.array(t), np.array(v_dist), label="Full P(r) Distribution", linewidth=3, color="black")

plt.xlabel("Time $t$ (µs)", fontsize=12)
plt.ylabel("DEER Signal $V(t)$", fontsize=12)
plt.title("Simulated DEER Time-Domain Traces", fontsize=14)
plt.legend(fontsize=12)
plt.tight_layout()
plt.show()

## 3. Rotamer Library Averaging

In structural biology, spin label flexibility is often modelled using rotamer libraries (e.g. MMM, MtsslWizard). The `deer_trace_rotamers` function can directly take two clouds of rotamers and their probabilities to generate a DEER trace without explicitly calculating the 1D $P(r)$ histogram first.

In [ ]:
from diff_epr.kernels import deer_trace_rotamers

# Synthetic rotamer clouds
key1, key2 = jax.random.split(jax.random.PRNGKey(101))

N_rot1 = 50
N_rot2 = 70

# Site 1 rotamers centered at origin
rotamers1 = 5.0 * jax.random.normal(key1, (N_rot1, 3))
weights1 = jnp.ones(N_rot1) / N_rot1

# Site 2 rotamers centered at [35.0, 0, 0]
rotamers2 = jnp.array([35.0, 0.0, 0.0]) + 6.0 * jax.random.normal(key2, (N_rot2, 3))
weights2 = jnp.ones(N_rot2) / N_rot2

# Simulate DEER directly from 3D rotamer clouds
v_rotamers = deer_trace_rotamers(
    rotamers1, weights1, rotamers2, weights2, time=t, modulation_depth=0.4, background_decay=0.1
)

plt.figure(figsize=(10, 4))
plt.plot(np.array(t), np.array(v_rotamers), color="purple", linewidth=2)
plt.xlabel("Time $t$ (µs)", fontsize=12)
plt.ylabel("DEER Signal $V(t)$", fontsize=12)
plt.title("DEER Trace Directly from 3D Rotamer Clouds", fontsize=14)
plt.tight_layout()
plt.show()